#### The regression model did not provide good accuracy, so we can consider using a boosting algorithm.

### boosting Algorithm

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [3]:
dataset=pd.read_csv("data/climate_change_impact_on_agriculture_2024.csv",index_col=None)

In [6]:
df=dataset

In [7]:
df.head()

,Year,Country,Region,Crop_Type,Average_Temperature_C,Total_Precipitation_mm,CO2_Emissions_MT,Crop_Yield_MT_per_HA,Extreme_Weather_Events,Irrigation_Access_%,Pesticide_Use_KG_per_HA,Fertilizer_Use_KG_per_HA,Soil_Health_Index,Adaptation_Strategies,Economic_Impact_Million_USD
0,2001,India,West Bengal,Corn,1.55,447.06,15.22,1.737,8,14.54,10.08,14.78,83.25,Water Management,808.13
1,2024,China,North,Corn,3.23,2913.57,29.82,1.737,8,11.05,33.06,23.25,54.02,Crop Rotation,616.22
2,2001,France,Ile-de-France,Wheat,21.11,1301.74,25.75,1.719,5,84.42,27.41,65.53,67.78,Water Management,796.96
3,2001,Canada,Prairies,Coffee,27.85,1154.36,13.91,3.890,5,94.06,14.38,87.58,91.39,No Adaptation,790.32
4,1998,India,Tamil Nadu,Sugarcane,2.19,1627.48,11.81,1.080,9,95.75,44.35,88.08,49.61,Crop Rotation,401.72


In [14]:
# Convert categorical features to dummy variables
df = pd.get_dummies(df, drop_first=True)

In [15]:
# Feature and Target selection
indep_X=df.iloc[:,[1,14]].values
dep_Y=df['Economic_Impact_Million_USD']

# Make Input X non Negative
#indep_X += abs(indep_X.min())  

In [16]:
# Split data into train and test
X_train,X_test,y_train,y_test = train_test_split(indep_X,dep_Y,test_size=0.30,random_state=0)

In [17]:
# GradientBoosting Regressor
from sklearn.ensemble import GradientBoostingRegressor
gb_model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)
gb_predictions = gb_model.predict(X_test)

In [18]:
#3.XGBoost Regressor
import xgboost as xgb
#model = xgb.XGBRegressor()
#from xgboost import XGBRegressor
xgb_model = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_predictions = xgb_model.predict(X_test)

In [19]:
# Step 4: Evaluate the Models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [20]:
# Function to Evaluate Models
def evaluate_model(y_test, predictions):
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    return {'MAE': mae, 'MSE': mse, 'R² Score': r2}

In [21]:
# Evaluate Gradient Boosting
gb_results = evaluate_model(y_test, gb_predictions)
print(f"\nGradient Boosting Results:\n MAE: {gb_results['MAE']}, MSE: {gb_results['MSE']}, R² Score: {gb_results['R² Score']}")


Gradient Boosting Results:
 MAE: 310.5707653511398, MSE: 148976.64102854457, R² Score: 0.1464830472265225


In [22]:
# Evaluate XGBoost
xgb_results = evaluate_model(y_test, xgb_predictions)
print(f"\nXGBoost Results:\n MAE: {xgb_results['MAE']}, MSE: {xgb_results['MSE']}, R² Score: {xgb_results['R² Score']}")


XGBoost Results:
 MAE: 311.1839265340169, MSE: 149307.01125491515, R² Score: 0.1445902901677505


### After hyper parameter tuning for GradientBoosting Regressor

In [23]:
# Import necessary libraries
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
}

# Initialize the Gradient Boosting model
gb_model = GradientBoostingRegressor()

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)

# Fit the model to the training data using GridSearchCV
grid_search.fit(X_train, y_train)

# Get the best parameters and best model
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Make predictions on the test set using the best model
y_pred_best = best_model.predict(X_test)

# Evaluate the best model
accuracy_best = r2_score(y_test, y_pred_best)

# Print the results
print( best_params)
print(accuracy_best)

{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
0.10090323388302569


### After hyper parameter tuning for XGBoost Regressor

In [24]:
# Various hyper-parameters to tune
import xgboost as xgb
from xgboost.sklearn import XGBRegressor
xgb1 = XGBRegressor()
parameters = {'nthread':[4], #when use hyperthread, xgboost may become slower
              'objective':['reg:linear'],
              'learning_rate': [.03, 0.05, .07], #so called `eta` value
              'max_depth': [5, 6, 7],
              'min_child_weight': [4],
              'silent': [1],
              'subsample': [0.7],
              'colsample_bytree': [0.7],
              'n_estimators': [500]}

xgb_grid = GridSearchCV(xgb1,
                        parameters,
                        cv = 2,
                        n_jobs = 5,
                        verbose=True)

xgb_grid.fit(X_train,
         y_train)

print(xgb_grid.best_score_)
print(xgb_grid.best_params_)

Fitting 2 folds for each of 9 candidates, totalling 18 fits
0.11849935490195362
{'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 500, 'nthread': 4, 'objective': 'reg:linear', 'silent': 1, 'subsample': 0.7}


## Conclusion

### The boosting algorithm model, even with hyperparameter tuning, did not provide good accuracy.